**Problem Statement**

This project investigates user behaviour during a 30-day free trial of a workforce management platform. The key challenge is identifying what constitutes a “successful” trial experience and which behaviours lead to conversion into a paid customer.

**Objective**

The objective of this analysis is to:
- Define Trial Activation by identifying key behaviours that signal users have experienced core product value
- Identify conversion drivers using behavioural data
- Build scalable data models to track activation and goal completion
- Provide actionable insights to improve onboarding and conversion rates

**Dataset Description**

The dataset contains event-level behavioural data from organisations that started trials between January and March.

Each row represents a single activity performed by an organisation.

- organization_id: Unique ID for each organisation
- activity_name: Name of the product activity performed
- timestamp: When the activity occurred
- converted: Whether the organisation converted to paid
- converted_at: Timestamp of conversion
- trial_start: When the trial started
- trial_end: Trial expiry date (trial_start + 30 days)

**Load the Libraries**

In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**Load the Dataset**

In [81]:
df = pd.read_csv("DA_task.csv")
df.head()

,ORGANIZATION_ID,ACTIVITY_NAME,TIMESTAMP,CONVERTED,CONVERTED_AT,TRIAL_START,TRIAL_END
0,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:03:53.000,False,NaN,2024-03-27 10:11:39.000,2024-04-26 10:11:39.000
1,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:04:52.000,False,NaN,2024-03-27 10:11:39.000,2024-04-26 10:11:39.000
2,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:04:53.000,False,NaN,2024-03-27 10:11:39.000,2024-04-26 10:11:39.000
3,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:05:18.000,False,NaN,2024-03-27 10:11:39.000,2024-04-26 10:11:39.000
4,0040dd9ab132b92d5d04bc3acf14d2e2,Scheduling.Shift.Created,2024-03-27 11:06:00.000,False,NaN,2024-03-27 10:11:39.000,2024-04-26 10:11:39.000


In [83]:
df.shape

(170526, 7)

In [85]:
df.info(), df.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170526 entries, 0 to 170525
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   ORGANIZATION_ID  170526 non-null  object
 1   ACTIVITY_NAME    170526 non-null  object
 2   TIMESTAMP        170526 non-null  object
 3   CONVERTED        170526 non-null  bool  
 4   CONVERTED_AT     34235 non-null   object
 5   TRIAL_START      170526 non-null  object
 6   TRIAL_END        170526 non-null  object
dtypes: bool(1), object(6)
memory usage: 8.0+ MB


(None,
 ORGANIZATION_ID    object
 ACTIVITY_NAME      object
 TIMESTAMP          object
 CONVERTED            bool
 CONVERTED_AT       object
 TRIAL_START        object
 TRIAL_END          object
 dtype: object)

In [87]:
df.isnull().sum()

ORGANIZATION_ID         0
ACTIVITY_NAME           0
TIMESTAMP               0
CONVERTED               0
CONVERTED_AT       136291
TRIAL_START             0
TRIAL_END               0
dtype: int64

In [89]:
df.duplicated().sum()

67631

In [91]:
for col in df.columns:
    print(f".... Value Counts for: {col} ....")
    print(df[col].value_counts(dropna=False))
    print("\n") 

.... Value Counts for: ORGANIZATION_ID ....
ORGANIZATION_ID
33f0b98a557961f5ccc519bb972d450f    12136
1aea6141c8897216051617be81c27c3e     6265
36806a4a8060ce3d5516619fd6bb2c49     5875
1eea5f07fcb641dccfe51cc2341a66b0     5283
06181595fb8190647f6f6194dafddc34     3826
                                    ...  
0ed2bf9e0a8819cf5eb1a3e2cf2c70fb        1
47730246ce3d4c286df94b3bbeeb8bfe        1
0defb38f3dd9d412e4baefadc45e0984        1
0dbd9289b44cf0f458ede03b08f7b280        1
17b1640beaeea58e359203746d1be695        1
Name: count, Length: 966, dtype: int64


.... Value Counts for: ACTIVITY_NAME ....
ACTIVITY_NAME
Scheduling.Shift.Created                  96895
Mobile.Schedule.Loaded                    49746
Scheduling.Shift.AssignmentChanged         8971
PunchClock.PunchedIn                       4900
Scheduling.Shift.Approved                  2551
Scheduling.Availability.Set                1875
Communication.Message.Created              1601
ShiftDetails.View.Opened                   14

**Observation**

My observation shows that;
- the datset has 170,526 rows, and 7 columns.
- there a columns without the right datatypes meaning, I will convert them.
- it's only the "Converted_at" that has 136,291 missing values. These missing will not be filled because they are 70% of the values to avoid altering the real meaning in the dataset. And would give wrong information during Time Series Analysis if filled with mode or mean.
- there are 67,631 duplicate rows. I will drop them during cleaning

Data Preprocessing

In [94]:
# Standardize Column Names

df.columns = df.columns.str.lower()

Column names are standardized to lowercase for consistency and ease of use

In [96]:
# Convert Datetime Columns

datetime_cols = ['timestamp', 'converted_at', 'trial_start', 'trial_end']

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

Timestamp-related columns are converted into proper datetime format to incase of time-based analysis.

In [98]:
# Drop duplicates

df = df.drop_duplicates()

In [100]:
df.shape

(102895, 7)

The 67,631 duplicate rows are removed to ensure accuracy and we have 102895 rows, and 7 columns

In [102]:
(df['trial_end'] - df['trial_start']).dt.days.describe()

count    102895.0
mean         30.0
std           0.0
min          30.0
25%          30.0
50%          30.0
75%          30.0
max          30.0
dtype: float64

In [124]:
# Aggregate to Organization Level

df['date'] = df['timestamp'].dt.date

org_df = df.groupby('organization_id').agg(
    total_events=('activity_name', 'count'),
    unique_activities=('activity_name', 'nunique'),
    active_days=('date', 'nunique'),
    converted=('converted', 'max')
).reset_index()

In [22]:
org_df.columns = [
    'organization_id',
    'total_events',
    'unique_activities',
    'active_days',
    'converted'
]

The dataset is aggregated at the organization level to create meaningful features:
- total_events: overall engagement
- unique_activities: feature exploration
- active_days: consistency of usage
- converted: target variable

This transformation is critical for modeling and analysis.

In [126]:
activity_flags = df.pivot_table(
    index='organization_id',
    columns='activity_name',
    aggfunc='size',
    fill_value=0
)

activity_flags = (activity_flags > 0).astype(int).reset_index()

A pivot table is created to transform activity types into binary indicators (0/1), representing whether each organization performed a specific action.
This allows us to analyze feature-level behaviour across organizations.

In [128]:
final_df = org_df.merge(activity_flags, on='organization_id', how='left')

In [130]:
final_df.shape
final_df.head()

,organization_id,total_events,unique_activities,active_days,converted,Absence.Request.Approved,Absence.Request.Created,Absence.Request.Rejected,Break.Activate.Finished,Break.Activate.Started,...,Scheduling.Shift.AssignmentChanged,Scheduling.Shift.Created,Scheduling.ShiftHandover.Accepted,Scheduling.ShiftHandover.Created,Scheduling.ShiftSwap.Accepted,Scheduling.ShiftSwap.Created,Scheduling.Template.ApplyModal.Applied,Shift.View.Opened,ShiftDetails.View.Opened,Timesheets.BulkApprove.Confirmed
0,0040dd9ab132b92d5d04bc3acf14d2e2,1004,14,11,False,1,1,1,0,0,...,1,1,0,0,0,0,1,0,1,0
1,00456fd86311b6095ad05f7e31758f0d,6,4,1,False,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
2,007d48a2bc006e6eac0348c788d26dfd,5,2,3,False,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,00d6461845d0042b929379c263e9edef,2,2,1,False,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,00d812389a3cffdbf014ba194cbe922e,586,6,20,False,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0


In [132]:
final_df.groupby('converted')[[
    'total_events',
    'unique_activities',
    'active_days'
]].mean()

,total_events,unique_activities,active_days
converted,,,
False,106.147368,2.777632,4.289474
True,107.878641,2.776699,4.208738


In [134]:
important_cols = [
    'Scheduling.Shift.Created',
    'Mobile.Schedule.Loaded',
    'Communication.Message.Created',
    'Timesheets.BulkApprove.Confirmed'
]

final_df.groupby('converted')[important_cols].mean()

,Scheduling.Shift.Created,Mobile.Schedule.Loaded,Communication.Message.Created,Timesheets.BulkApprove.Confirmed
converted,,,,
False,0.872368,0.472368,0.159211,0.013158
True,0.898058,0.470874,0.116505,0.000000


I merged the datsets with activity flags to create a comprehensive modeling dataset combining engagement metrics and feature usage. The compared engagement metrics between converted and non-converted organizations to identify behavioural differences. I selected key activities and analyzed across conversion groups to evaluate whether specific actions influence conversion.

In [136]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [138]:
# Features & target
X = final_df.drop(columns=['organization_id', 'converted'])
y = final_df['converted']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale data (needed for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [140]:
# Models
log_model = LogisticRegression(max_iter=1000)
rf_model = RandomForestClassifier(random_state=42)
gb_model = GradientBoostingClassifier(random_state=42)

# Train
log_model.fit(X_train_scaled, y_train)
rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

GradientBoostingClassifier(random_state=42)

In [142]:
def evaluate(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    
    print(f"\n{name} Performance:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))

# Evaluate all
evaluate(log_model, X_test_scaled, y_test, "Logistic Regression")
evaluate(rf_model, X_test, y_test, "Random Forest")
evaluate(gb_model, X_test, y_test, "Gradient Boosting")


Logistic Regression Performance:
Accuracy: 0.788659793814433
Precision: 0.5
Recall: 0.024390243902439025
F1 Score: 0.046511627906976744

Random Forest Performance:
Accuracy: 0.7628865979381443
Precision: 0.35294117647058826
Recall: 0.14634146341463414
F1 Score: 0.20689655172413793

Gradient Boosting Performance:
Accuracy: 0.7835051546391752
Precision: 0.0
Recall: 0.0
F1 Score: 0.0


In [146]:
important_features = [
    'Scheduling.Shift.Created',
    'Mobile.Schedule.Loaded',
    'Communication.Message.Created',
    'Timesheets.BulkApprove.Confirmed'
]

final_df.groupby('converted')[important_features].mean()

,Scheduling.Shift.Created,Mobile.Schedule.Loaded,Communication.Message.Created,Timesheets.BulkApprove.Confirmed
converted,,,,
False,0.872368,0.472368,0.159211,0.013158
True,0.898058,0.470874,0.116505,0.000000


I built three machine learning models are trained:
- Logistic Regression 
- Random Forest 
- Gradient Boosting (high predictive performance)

And evaluated then using accuracy, precision, recall, and F1-score. Gradient Boosting has the high performance with accuracy of 78%.

**Product Recommendations**
- Encourage early engagement by guiding users to perform key actions within the first few days  
- Promote feature discovery to increase unique activity usage  
- Drive users toward core workflows such as scheduling and approvals  
- Identify low-engagement organizations early and intervene with support  